# 1 下载日历数据

In [1]:
apple_cn_ics_path = '../source_ics/apple_cn.ics'
apple_us_ics_path = '../source_ics/apple_us.ics'
google_ics_path = '../source_ics/google.ics'

custom_ics_path = './apple_supplement.ics'

In [2]:
import requests

# 要下载的日历 URL
google_calendar_url = 'https://calendar.google.com/calendar/ical/zh-cn.china%23holiday%40group.v.calendar.google.com/public/basic.ics'
apple_calendar_cn_url = 'https://calendars.icloud.com/holidays/cn_zh.ics'
apple_calendar_us_url = 'https://calendars.icloud.com/holidays/us_en-us.ics'

# 下载日历
for url in google_calendar_url, apple_calendar_cn_url, apple_calendar_us_url:
    # 发送 HTTP GET 请求
    response = requests.get(url)

    # 检查请求是否成功
    if response.status_code == 200:
        # 根据URL选择文件名
        file_name = ''
        if url == google_calendar_url:
            file_name = '../source_ics/google.ics'
        elif url == apple_calendar_cn_url:
            file_name = '../source_ics/apple_cn.ics'
        elif url == apple_calendar_us_url:
            file_name = '../source_ics/apple_us.ics'

        # 将网页内容保存到文件中
        if file_name:
            with open(file_name, 'w', encoding='utf-8') as f:
                f.write(response.text)
            print(f'网页已成功下载并保存为 {file_name}')
        
    else:
        print(f'请求失败，状态码：{response.status_code}')


网页已成功下载并保存为 ../source_ics/google.ics
网页已成功下载并保存为 ../source_ics/apple_cn.ics
网页已成功下载并保存为 ../source_ics/apple_us.ics


# 2 创建补充日历
创建日历, 并且添加 '教师节', '情人节', '父亲节', '母亲节'

In [35]:
from icalendar import prop, Calendar, Event
from datetime import date
import hashlib
import re

# 创建一个新的日历对象
custom_calendar = Calendar()

# 添加日历属性
custom_calendar.add('VERSION', '2.0')
custom_calendar.add('PRODID', 'icalendar-ruby')
custom_calendar.add('CALSCALE', 'GREGORIAN')
custom_calendar.add('X-WR-CALNAME', '中国大陆节假日补充')
custom_calendar.add('X-APPLE-LANGUAGE', 'zh')
custom_calendar.add('X-APPLE-REGION', 'CN')

# 添加教师节事件
event_teachers_day = Event()
event_teachers_day.add('DTSTAMP;VALUE', "DATE:{date.today().strftime('%Y%m%d')}")
event_teachers_day.add('SUMMARY;LANGUAGE', 'zh_CN:教师节')
event_teachers_day.add('DTSTART;VALUE', f'DATE:{date.today().year-5}0910')
event_teachers_day.add('RRULE:FREQ', 'YEARLY;COUNT=10')
event_teachers_day.add('CLASS', 'PUBLIC')
event_teachers_day.add('TRANSP', 'TRANSPARENT')
event_teachers_day.add('CATEGORIES', '节慶')
event_teachers_day.add('UID', f'{hashlib.md5(str(event_teachers_day).encode()).hexdigest()}@jayden')
custom_calendar.add_component(event_teachers_day)

# 添加 '情人节' 事件
event_valentines_day = Event()
event_valentines_day.add('DTSTAMP;VALUE', "DATE:{date.today().strftime('%Y%m%d')}")
event_valentines_day.add('SUMMARY;LANGUAGE', 'zh_CN:情人节')
event_valentines_day.add('DTSTART;VALUE', f'DATE:{date.today().year-5}0214')
event_valentines_day.add('RRULE:FREQ', 'YEARLY;COUNT=10')
event_valentines_day.add('CLASS', 'PUBLIC')
event_valentines_day.add('TRANSP', 'TRANSPARENT')
event_valentines_day.add('CATEGORIES', '节慶')
event_valentines_day.add('UID', f'{hashlib.md5(str(event_valentines_day).encode()).hexdigest()}@jayden')
custom_calendar.add_component(event_valentines_day)

# 添加 '父亲节', '母亲节' 事件
with open(apple_us_ics_path, 'rb') as f:
    apple_us_calendar = Calendar.from_ical(f.read())
    for event in apple_us_calendar.walk('VEVENT'):
        if event.get('SUMMARY') == 'Father’s Day':
            summary = prop.vText('父亲节')
            summary.params['LANGUAGE'] = 'zh_CN'
            event['SUMMARY'] = summary
            dtstamp = prop.vDDDTypes(date.today())
            dtstamp.params['VALUE'] = 'DATE'
            event['DTSTAMP'] = dtstamp
            event['CATEGORIES'] = '节慶'
            del event['UID']
            del event['X-APPLE-UNIVERSAL-ID']
            event['UID'] = f'{hashlib.md5(str(event).encode()).hexdigest()}@jayden'
            custom_calendar.add_component(event)
        elif event.get('SUMMARY') == 'Mother’s Day':
            summary = prop.vText('母亲节')
            summary.params['LANGUAGE'] = 'zh_CN'
            event['SUMMARY'] = summary
            dtstamp = prop.vDDDTypes(date.today())
            dtstamp.params['VALUE'] = 'DATE'
            event['DTSTAMP'] = dtstamp
            event['CATEGORIES'] = '节慶'
            del event['UID']
            del event['X-APPLE-UNIVERSAL-ID']
            event['UID'] = f'{hashlib.md5(str(event).encode()).hexdigest()}@jayden'
            custom_calendar.add_component(event)

# 将日历对象转换为字符串
ical_str = custom_calendar.to_ical().decode('utf-8')

# 使用正则表达式只替换特定属性的分隔符
ical_str = re.sub(r'(DTSTAMP;VALUE|SUMMARY;LANGUAGE|DTSTART;VALUE|DTEND;VALUE|RRULE:FREQ):', r'\1=', ical_str)

# 将内容写入 .ics 文件
with open(custom_ics_path, 'wb') as f:
    f.write(ical_str.encode('utf-8'))

print("创建新的的 .ics 文件已保存为 %s" % custom_ics_path)


创建新的的 .ics 文件已保存为 ./apple_supplement.ics
